# 61. Mixed Cartesian + arbitrary SO(3) rotation fine-tuning | 6D regression

**Goal.** Continue from the complete Notebook 60 **002.30 best checkpoint** (Transformer, pooler, MLP and 6D head) while training on a 50/50 per-sample mixture of (i) signed single-axis X/Y/Z faults and (ii) arbitrary isotropic 3D rotation axes. Exactly one of the three sensors is faulted per non-identity example. Preserve the Notebook 60 direct 6D rotation loss and its physically consistent true-frame augmentation.

**Workflow.** Run the import/data/model/sampling/probe cells; run one **reversible stochastic preflight**; inspect its plots; enable the **full training** switch; evaluate **002.30 versus 61 best** on matched Cartesian and arbitrary-axis one-step tests and chronological 12-window feedback. A new Notebook 61 run uses fresh AdamW states; resuming *that same Notebook 61 run* restores the optimizer/scheduler/RNG state from `last.pt`. Never resume 002.30's AdamW into a different augmentation experiment.

No additional repository modifications required, assuming Notebook 60's `rotation_axes` augmentation extension is already installed. Data and checkpoints stay in your repository. Test sequences Urban38/39 are **not** used here.

## 1. Imports and repository

In [ ]:
from pathlib import Path
from dataclasses import asdict, replace
from collections import defaultdict
import gc
import math
import random
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from IPython.display import display

ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / 'src' / 'obscalib').is_dir():
        ROOT = candidate
        break
else:
    raise FileNotFoundError('Run from the nn-obs-calib repository or a descendant directory.')
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from kaist_dataset.dataset import KAISTCalibrationDataset, KAISTRigCalibration, build_kaist_sequence_specs, make_kaist_dataloader
from obscalib.augmentations import PerturbationDistribution
from obscalib.augmentations.config import AugmentationConfig, CalibrationEventConfig, CommonBodyFrameRandomizationConfig, FrameRandomizationConfig, NoiseAugmentationConfig, PerturbationMagnitudeConfig, PriorPerturbationConfig, SamplingRateAugmentationConfig, SensorTopologyAugmentationConfig, TrueCalibrationAugmentationConfig
from obscalib.augmentations.pipeline import AugmentationPipeline
from obscalib.augmentations.prior_perturbation import CalibrationPriorPerturber
from obscalib.config import CalibrationPoolerConfig, MLPConfig, PooledCalibrationHeadConfig, PooledCalibrationModelConfig, TransformerConfig, WindowingConfig
from obscalib.data.collation import collate_windows
from obscalib.data.device import window_batch_to
from obscalib.data.structures import WindowBatch
from obscalib.geometry.lie import so3_exp
from obscalib.geometry.processing import GeometryProcessor
from obscalib.geometry.rotation_6d import matrix_to_rotation_6d, rotation_6d_to_matrix
from obscalib.models.pooled_calibration_6d_model import PooledCalibration6DModel, load_6d_pretrained_features
from obscalib.calibration.rotation_6d_update import apply_calibration_6d_update
from obscalib.training.rotation_6d_loss import Calibration6DLossWeights, calibration_6d_loss
from obscalib.tokenization.tokenizer import Tokenizer
from obscalib.training.checkpoint import load_training_checkpoint, save_training_checkpoint

print('Repository:', ROOT, '| Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())


## 2. Settings: initialization, sampling and execution switches

In [ ]:
# NOTE: this run is separate from Notebook 60's run. Do not overwrite 002.30 checkpoints.
RUN_NAME = 'kaist_pooled_calibration_6d_002.31_cartesian_arbitrary_finetune'
RUN_DIR = ROOT / 'runs' / RUN_NAME
SOURCE_230_PATH = ROOT / 'runs' / 'kaist_pooled_calibration_6d_002.30_all_sensors_single_axis_xyz' / 'best.pt'
SOURCE_227_PATH = ROOT / 'runs' / 'kaist_pooled_calibration_6d_002.27_full_rotation_regression' / 'best.pt'

RANDOM_SEED, TRAIN_AUG_SEED, VAL_AUG_SEED = 12345, 23456, 34567
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE = torch.float32
CALIBRATION_KEYS = ('imu', 'lidar', 'lidar_right')
WINDOW_DURATION_S, WINDOW_STRIDE_S, MAX_SAMPLES_PER_SENSOR = 5.0, 5.0, 700
MEASUREMENT_DIM, TOKEN_INPUT_DIM = 6, 8
D_MODEL, N_HEADS, NUM_TRANSFORMER_LAYERS, DIM_FEEDFORWARD = 128, 8, 12, 512
TRANSFORMER_DROPOUT, NUM_SUMMARY_TOKENS = 0.0, 1
POOLER_N_HEADS, POOLER_DROPOUT = 8, 0.0
MLP_HIDDEN_DIMS, SHARED_FEATURE_DIM, MLP_DROPOUT = (256, 128), 128, 0.0
HEAD_HIDDEN_DIMS, HEAD_DROPOUT = (128, 64), 0.0
REFINED_TAU_BY_SENSOR_S = {'imu': 0.0, 'lidar': -0.025, 'lidar_right': -0.035}
RANDOMIZE_TRUE_SENSOR_FRAMES = True
TRUE_CALIBRATION_ROTATION_MAX_RAD = np.deg2rad(180.0)
STUDENT_T_DOF = 3
FINE_ROTATION_MAX_DEG, FINE_ROTATION_SCALE_DEG = 60.0, 30.0
COARSE_ROTATION_MIN_DEG, COARSE_ROTATION_MAX_DEG = 60.0, 179.0
COARSE_PRIOR_PROBABILITY = 0.50
IDENTITY_PROBABILITY = 0.10
CARTESIAN_PROBABILITY = 0.50  # Among non-identity cases; the other half have random 3D axes.
ACTIVE_ROTATION_SUPERVISION_WEIGHT = 0.75
LOSS_WEIGHTS = Calibration6DLossWeights(rotation_weight=2.0, rotation_6d_weight=1.0, rotation_chordal_weight=0.0, translation_weight=0.0, time_offset_weight=0.0, translation_scale_m=0.05, time_offset_scale_s=0.05)

WEIGHT_DECAY, GRADIENT_CLIP_NORM = 1e-4, 1.0
HEAD_MLP_LR, TRANSFORMER_LR = 1e-4, 1e-5
TRAIN_BATCH_SIZE, NUM_WORKERS, GRADIENT_ACCUMULATION_STEPS = 32, 0, 1
LR_REDUCE_FACTOR, LR_PATIENCE_PROBES = 0.5, 6
FULL_TARGET_EPOCHS = 40  # TOTAL number of epochs, including already completed epochs when resuming.
PROBE_EVERY_EPOCHS = 2
MAX_TRAIN_BATCHES_PER_EPOCH, MAX_VAL_BATCHES_PER_EPOCH = None, None
SHOW_PLOTS = True

# Run preflight, inspect its plots, then set RUN_FULL_TRAINING=True and rerun the full setup/loop.
RUN_STOCHASTIC_PREFLIGHT = True
PREFLIGHT_STEPS, PREFLIGHT_BATCH_SIZE, PREFLIGHT_REPORT_EVERY = 700, 32, 100
PREFLIGHT_HEAD_LR, PREFLIGHT_TRANSFORMER_LR = HEAD_MLP_LR, TRANSFORMER_LR
PREFLIGHT_PROBE_WINDOWS, PREFLIGHT_PROBE_BATCH_SIZE, PREFLIGHT_3D_DIRECTIONS = 12, 12, 2
RUN_FULL_TRAINING = False
RESUME_FULL_TRAINING = False  # True only to resume Notebook 61's own last.pt with the SAME configuration.
FULL_INIT_FROM_PREFLIGHT_BEST = False  # Otherwise initialize full training from 002.30 best.pt.

PROBE_ANGLES_DEG = (0.0, 30.0, 60.0, 90.0, 120.0, 150.0, 175.0, 179.0)
PROBE_SCORE_ANGLES_DEG = (30.0, 60.0, 90.0, 120.0, 150.0, 175.0, 179.0)
FULL_PROBE_WINDOWS, FULL_PROBE_BATCH_SIZE, FULL_PROBE_3D_DIRECTIONS = 16, 16, 4
PROBE_SEED = VAL_AUG_SEED + 5_200_000
ZERO_DRIFT_WEIGHT, WORST_PAIR_WEIGHT, NEAR_PI_WEIGHT = 0.15, 0.25, 0.10

EVAL_RUN = False  # Set True AFTER the full run produced best.pt; then execute the evaluation section.
EVAL_INCLUDE_227 = False  # Optional additional checkpoint; main comparison always uses 002.30 and 61 best.
EVAL_WINDOWS, EVAL_BATCH_SIZE, EVAL_3D_DIRECTIONS = 32, 8, 4
EVAL_ANGLES_DEG = (0.0, 15.0, 30.0, 60.0, 90.0, 120.0, 150.0, 175.0, 179.0)
EVAL_ROLLOUT_SEQUENCES = ('Urban26', 'Urban28')
EVAL_ROLLOUT_STARTS, EVAL_ROLLOUT_TRUTHS, EVAL_ROLLOUT_STEPS = 1, 1, 12
EVAL_ROLLOUT_ANGLES = (0.0, 30.0, 90.0, 150.0, 179.0)
EVAL_ROLLOUT_3D_DIRECTIONS = 3

assert SOURCE_230_PATH.is_file(), f'Notebook 60 002.30 best checkpoint is missing: {SOURCE_230_PATH}'
assert 0.0 < CARTESIAN_PROBABILITY < 1.0 and 0.0 <= IDENTITY_PROBABILITY < 1.0
assert 0.0 <= COARSE_PRIOR_PROBABILITY <= 1.0 and COARSE_ROTATION_MAX_DEG < 180

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(RANDOM_SEED)
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Notebook 61:', RUN_DIR)
print('Full model initialization:', SOURCE_230_PATH)
print('Sampling: 10% identity; otherwise 50% Cartesian and 50% arbitrary axes; one random sensor')


## 3. KAIST data: exactly the same splits and physical streams as Notebook 60

In [ ]:
KAIST_DATASET_ROOT = Path("/mnt/d/Downloads/MobRobLab/KAISTDataset")
LIDAR_POSE_ROOT = Path("/home/camel/Skoltech/phd_proposal/data/KAISTDataset")

TRAIN_NAMES = ["Urban08", "Urban12", "Urban13", "Urban14", "Urban15", "Urban16"]
VAL_NAMES = ["Urban26", "Urban28"]
TEST_NAMES = ["Urban38", "Urban39"]
ALL_SEQUENCE_NAMES = TRAIN_NAMES + VAL_NAMES + TEST_NAMES

if len(ALL_SEQUENCE_NAMES) != len(set(ALL_SEQUENCE_NAMES)):
    raise ValueError("Train, validation, and test sequence sets overlap.")

T_B_L = np.array([
    [-0.514066, -0.702201, -0.492595, -0.440699],
    [0.486485, -0.711672, 0.506809, 0.397052],
    [-0.706447, 0.0208933, 0.707457, 1.90953],
    [0.0, 0.0, 0.0, 1.0],
], dtype=float)

# Right LiDAR -> body, in the same convention as T_B_L for the left LiDAR.
T_B_L_right = np.array([
    [-0.512152,  0.699241, -0.498761, -0.449885],
    [-0.494811, -0.714859, -0.494104, -0.416713],
    [-0.702041, -0.0062642, 0.712109,  1.91294],
    [ 0.0,       0.0,       0.0,       1.0],
], dtype=float)

T_I_B = np.array([
    [1.0, 0.0, 0.0, -0.07],
    [0.0, 1.0, 0.0, 0.0],
    [0.0, 0.0, 1.0, 1.7],
    [0.0, 0.0, 0.0, 1.0],
], dtype=float)

rig_calibration = KAISTRigCalibration(T_B_I=np.linalg.inv(T_I_B), T_B_L=T_B_L, tau_imu_s=KAIST_TAU_IMU_S, tau_lidar_s=KAIST_TAU_LIDAR_S, T_B_L_right=T_B_L_right, tau_lidar_right_s=KAIST_TAU_LIDAR_RIGHT_S)
windowing_config = WindowingConfig(window_duration_s=WINDOW_DURATION_S, window_stride_s=WINDOW_STRIDE_S, max_samples_per_sensor=MAX_SAMPLES_PER_SENSOR)

print("train:", TRAIN_NAMES)
print("validation:", VAL_NAMES)
print("test:", TEST_NAMES)
print("Corrected rig τ [ms]:", {key: 1000.0 * value for key, value in REFINED_TAU_BY_SENSOR_S.items()})


In [ ]:
all_specs = build_kaist_sequence_specs(sequence_names=ALL_SEQUENCE_NAMES, kaist_dataset_root=KAIST_DATASET_ROOT, lidar_pose_root=LIDAR_POSE_ROOT, lidar_right_pose_filename="lidar_map_poses_vlp_right.csv")

# The 3-sensor experiment must never silently fall back to 2 sensors.
for spec in all_specs:
    if spec.lidar_right_pose_csv is None or not spec.lidar_right_pose_csv.is_file():
        raise FileNotFoundError(f"Missing required right-LiDAR pose CSV for {spec.name}: {spec.lidar_right_pose_csv}")
spec_by_name = {spec.name: spec for spec in all_specs}

train_dataset = KAISTCalibrationDataset([spec_by_name[name] for name in TRAIN_NAMES], rig_calibration, windowing_config, measurement_dtype=torch.float32, source_timestamp_dtype=torch.float64, window_timestamp_dtype=torch.float32)
val_dataset = KAISTCalibrationDataset([spec_by_name[name] for name in VAL_NAMES], rig_calibration, windowing_config, measurement_dtype=torch.float32, source_timestamp_dtype=torch.float64, window_timestamp_dtype=torch.float32)
test_dataset = KAISTCalibrationDataset([spec_by_name[name] for name in TEST_NAMES], rig_calibration, windowing_config, measurement_dtype=torch.float32, source_timestamp_dtype=torch.float64, window_timestamp_dtype=torch.float32)

for split_name, dataset in (("train", train_dataset), ("validation", val_dataset), ("test", test_dataset)):
    if tuple(dataset.calibration_keys) != CALIBRATION_KEYS:
        raise AssertionError(f"{split_name} dataset has calibration keys {dataset.calibration_keys}, expected {CALIBRATION_KEYS}")

print(train_dataset.summary())
print()
print(val_dataset.summary())
print()
print(test_dataset.summary())

# Check the actual WindowBatch ground truth, not just constructor arguments.
for split_name, split_dataset in (("train", train_dataset), ("validation", val_dataset), ("test", test_dataset)):
    sample = collate_windows([split_dataset[0]])
    for sensor, expected_s in REFINED_TAU_BY_SENSOR_S.items():
        observed = float(sample.current_calibration[sensor].time_offset.reshape(-1)[0].item())
        assert abs(observed - expected_s) < 2e-6, f"{split_name}/{sensor}: dataset τ={observed} but expected {expected_s}"
    print(f"{split_name}: τ targets [ms] verified:", {sensor: round(float(state.time_offset.reshape(-1)[0]) * 1000, 3) for sensor, state in sample.current_calibration.items()})

# Compare the zero-clock and corrected-clock dataset instances on the same held-out
# physical windows. This checks whether the loader alters raw measurements when the
# rig reference changes; it never deliberately shifts raw timestamps in this notebook.
zero_clock_rig = KAISTRigCalibration(T_B_I=np.linalg.inv(T_I_B), T_B_L=T_B_L, tau_imu_s=0.0, tau_lidar_s=0.0, T_B_L_right=T_B_L_right, tau_lidar_right_s=0.0)
zero_clock_control = KAISTCalibrationDataset([spec_by_name[TEST_NAMES[0]]], zero_clock_rig, windowing_config, measurement_dtype=torch.float32, source_timestamp_dtype=torch.float64, window_timestamp_dtype=torch.float32)
corrected_control = KAISTCalibrationDataset([spec_by_name[TEST_NAMES[0]]], rig_calibration, windowing_config, measurement_dtype=torch.float32, source_timestamp_dtype=torch.float64, window_timestamp_dtype=torch.float32)
if len(zero_clock_control) != len(corrected_control):
    raise RuntimeError("Changing calibration τ modified the number of windows. Inspect dataset timestamp processing before training.")
for index in sorted({0, len(zero_clock_control)//2, len(zero_clock_control)-1}):
    old, new = zero_clock_control[index], corrected_control[index]
    for stream_name in old.streams:
        old_stream, new_stream = old.streams[stream_name], new.streams[stream_name]
        torch.testing.assert_close(old_stream.values, new_stream.values, rtol=0, atol=0)
        torch.testing.assert_close(old_stream.timestamps, new_stream.timestamps, rtol=0, atol=0)
        if old_stream.interval_start_timestamps is not None:
            torch.testing.assert_close(old_stream.interval_start_timestamps, new_stream.interval_start_timestamps, rtol=0, atol=0)
print("PASS: refined τ changes calibration reference while preserving native stream measurements and timestamps.")
del zero_clock_control, corrected_control


In [ ]:
train_shuffle_generator = torch.Generator().manual_seed(RANDOM_SEED)
train_loader = make_kaist_dataloader(train_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True, drop_last=True, num_workers=NUM_WORKERS, generator=train_shuffle_generator)
val_loader = make_kaist_dataloader(val_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=False, drop_last=False, num_workers=NUM_WORKERS)
print('Train/validation windows:', len(train_dataset), len(val_dataset))
print('Batches:', len(train_loader), len(val_loader))


## 4. Physical augmentation and canonical calibration conventions

In [ ]:
true_calibration_config = AugmentationConfig(
    sensor_topology=SensorTopologyAugmentationConfig(enabled=False),
    sampling_rate=SamplingRateAugmentationConfig(enabled=False),
    frame_randomization=FrameRandomizationConfig(enabled=False),
    common_body_frame=CommonBodyFrameRandomizationConfig(enabled=False),
    true_calibration=TrueCalibrationAugmentationConfig(enabled=RANDOMIZE_TRUE_SENSOR_FRAMES, probability=1.0, rotation_probability=1.0, translation_probability=0.0, time_offset_probability=0.0, rotation=PerturbationMagnitudeConfig(maximum=TRUE_CALIBRATION_ROTATION_MAX_RAD)),
    prior_perturbation=PriorPerturbationConfig(enabled=False),
    calibration_event=CalibrationEventConfig(enabled=False),
    noise=NoiseAugmentationConfig(enabled=False),
)
physical_augmenter = AugmentationPipeline(true_calibration_config)

def augmentation_generator(seed):
    return torch.Generator(device=DEVICE.type).manual_seed(int(seed))

def clone_with_current_calibration(window, states):
    return WindowBatch(streams=dict(window.streams), current_calibration=states, metadata=dict(window.metadata), targets=window.targets, dataset_indices=window.dataset_indices)

def apply_independent_prior_faults(states, xi_by_key):
    """Left-multiply rotation ONLY, leaving lever arms, time offsets and healthy sensors untouched."""
    updated = {}
    for key, state in states.items():
        prior = state.transform.clone()
        prior[:, :3, :3] = so3_exp(xi_by_key[key][:, :3]) @ state.transform[:, :3, :3]
        updated[key] = replace(state, transform=prior)
    return updated

print('Physical augmentation matches Notebook 60: randomized true frames, no artificial sensor noise.')


## 5. Model, complete 002.30 weights and unchanged Notebook 60 rotation loss

In [ ]:
model_config = PooledCalibrationModelConfig(
    transformer=TransformerConfig(input_dim=TOKEN_INPUT_DIM, d_model=D_MODEL, n_heads=N_HEADS, num_layers=NUM_TRANSFORMER_LAYERS, dim_feedforward=DIM_FEEDFORWARD, dropout=TRANSFORMER_DROPOUT, num_summary_tokens=NUM_SUMMARY_TOKENS, activation='gelu'),
    mlp=MLPConfig(hidden_dims=MLP_HIDDEN_DIMS, output_dim=SHARED_FEATURE_DIM, activation='gelu', dropout=MLP_DROPOUT),
    pooler=CalibrationPoolerConfig(n_heads=POOLER_N_HEADS, dropout=POOLER_DROPOUT, calibration_context_dim=0),
    head=PooledCalibrationHeadConfig(calibration_context_dim=0, hidden_dims=HEAD_HIDDEN_DIMS, activation='gelu', dropout=HEAD_DROPOUT),
)
geometry_processor = GeometryProcessor()
tokenizer = Tokenizer(measurement_dim=MEASUREMENT_DIM, calibration_keys=CALIBRATION_KEYS)
model = PooledCalibration6DModel(model_config).to(device=DEVICE, dtype=DTYPE)
load_training_checkpoint(SOURCE_230_PATH, model=model, map_location=DEVICE, restore_rng_state=False)
model.freeze_feature_extractor(False)
model.head_6d.requires_grad_(True)
assert all(p.requires_grad for p in model.parameters())

def model_predictions(window, *, model_override=None):
    if tuple(window.current_calibration) != CALIBRATION_KEYS:
        raise ValueError(f'Unexpected key order: {tuple(window.current_calibration)}')
    canonical = geometry_processor(window.streams, window.metadata, window.current_calibration)
    tokens = tokenizer(canonical, calibration_keys=tuple(window.current_calibration))
    return (model if model_override is None else model_override)(tokens)

def parameter_groups(module, *, transformer_lr=TRANSFORMER_LR, other_lr=HEAD_MLP_LR):
    transformer = list(module.transformer_encoder.parameters())
    other = list(module.pooler.parameters()) + list(module.mlp_body.parameters()) + list(module.head_6d.parameters())
    assert len({id(p) for p in transformer + other}) == len(list(module.parameters()))
    return [dict(params=transformer, lr=transformer_lr, name='transformer'), dict(params=other, lr=other_lr, name='pooler_mlp_head')]

print('Loaded COMPLETE 002.30 model, including 6D head. Parameters:', sum(p.numel() for p in model.parameters()))


In [ ]:
def rotation_geodesic_deg(R_first, R_second):
    """Numerically stable geodesic SO(3) angle, including near 180°. Shape [...]."""
    relative = R_first @ R_second.transpose(-1, -2)
    skew = torch.stack((relative[..., 2, 1] - relative[..., 1, 2], relative[..., 0, 2] - relative[..., 2, 0], relative[..., 1, 0] - relative[..., 0, 1]), dim=-1)
    sine = 0.5 * torch.linalg.vector_norm(skew, dim=-1)
    cosine = 0.5 * (relative.diagonal(dim1=-2, dim2=-1).sum(dim=-1) - 1.0)
    return torch.rad2deg(torch.atan2(sine, cosine))


def compute_6d_objective(window, output):
    """Joint 6D+chordal rotation loss, translation/tau loss, and active-fault emphasis."""
    if window.targets is None:
        raise ValueError('Training requires full GT targets')
    supervised = calibration_6d_loss(output.predictions, window.current_calibration, window.targets, weights=LOSS_WEIGHTS)
    active_terms, active_angle_errors, active_predicted_angles, active_target_angles = [], [], [], []
    for key in CALIBRATION_KEYS:
        state, truth, pred = window.current_calibration[key], window.targets[key], output.predictions[key]
        R_target = truth.next_transform[:, :3, :3] @ state.transform[:, :3, :3].transpose(-1, -2)
        angle_before = rotation_geodesic_deg(state.transform[:, :3, :3], truth.next_transform[:, :3, :3])
        active = angle_before > 1e-3
        if bool(active.any()):
            target_6d = matrix_to_rotation_6d(R_target)
            reg = (pred.rotation_6d - target_6d).square().mean(dim=-1)
            chord = 0.5 * (pred.delta_rotation - R_target).square().sum(dim=(-2, -1))
            active_terms.append((LOSS_WEIGHTS.rotation_6d_weight * reg + LOSS_WEIGHTS.rotation_chordal_weight * chord)[active])
            active_angle_errors.append(rotation_geodesic_deg(pred.delta_rotation[active], R_target[active]).detach())
            active_predicted_angles.append(rotation_geodesic_deg(pred.delta_rotation[active], torch.eye(3, device=DEVICE, dtype=DTYPE).expand_as(pred.delta_rotation[active])).detach())
            active_target_angles.append(angle_before[active].detach())
    extra_active = LOSS_WEIGHTS.rotation_weight * torch.cat(active_terms).mean() if active_terms else supervised.total * 0.0
    loss = supervised.total + ACTIVE_ROTATION_SUPERVISION_WEIGHT * extra_active
    with torch.no_grad():
        corrected = apply_calibration_6d_update(window.current_calibration, output.predictions)
        values = {}
        for key in CALIBRATION_KEYS:
            target = window.targets[key]
            state = corrected[key]
            values[f'{key}/rotation_error_deg'] = rotation_geodesic_deg(state.transform[:, :3, :3], target.next_transform[:, :3, :3]).mean().item()
            values[f'{key}/translation_error_cm'] = (100.0 * torch.linalg.vector_norm(state.transform[:, :3, 3] - target.next_transform[:, :3, 3], dim=-1)).mean().item()
            values[f'{key}/tau_error_ms'] = (1000.0 * (state.time_offset - target.next_time_offset).abs()).mean().item()
        values.update({'loss_total': float(loss.detach()), 'loss_supervised': float(supervised.total.detach()), 'loss_active_rotation': float(extra_active.detach()), 'loss_rotation_6d': float(supervised.rotation_6d.detach()), 'loss_rotation_chordal': float(supervised.rotation_chordal.detach()), 'loss_translation': float(supervised.translation.detach()), 'loss_tau': float(supervised.time_offset.detach()), 'active_rotation_error_deg': torch.cat(active_angle_errors).mean().item() if active_angle_errors else 0.0, 'active_target_deg': torch.cat(active_target_angles).mean().item() if active_target_angles else 0.0, 'active_predicted_step_deg': torch.cat(active_predicted_angles).mean().item() if active_predicted_angles else 0.0, 'active_rotation_count': sum(x.numel() for x in active_angle_errors)})
    return loss, values


def metric_average(sums, counts):
    return {key: sums[key] / counts[key] for key in sums}


## 6. Mixed Cartesian/arbitrary 3D sampler and sanity checks

`one_rotation_axis_per_sample=True` samples one signed coordinate axis. `False` samples an isotropic unit axis in ℝ³. Both use the same bounded fine/coarse magnitude distributions and randomly select **one sensor per example**. The mixing decision is made per sample, not once per batch.

In [ ]:
def prior_config_61(*, arbitrary, coarse):
    magnitude = (PerturbationMagnitudeConfig(maximum=np.deg2rad(COARSE_ROTATION_MAX_DEG), minimum=np.deg2rad(COARSE_ROTATION_MIN_DEG), distribution=PerturbationDistribution.UNIFORM)
                 if coarse else PerturbationMagnitudeConfig(maximum=np.deg2rad(FINE_ROTATION_MAX_DEG), scale=np.deg2rad(FINE_ROTATION_SCALE_DEG), degrees_of_freedom=STUDENT_T_DOF, distribution=PerturbationDistribution.TRUNCATED_STUDENT_T))
    return PriorPerturbationConfig(enabled=True, probability=1.0, probability_by_key={}, rotation=magnitude, translation=None, time_offset=None, one_calibration_per_sample=True, one_rotation_axis_per_sample=not arbitrary, rotation_axes=(True, True, True), one_perturbation_type_per_sample=False)

prior_samplers_61 = {
    (arbitrary, coarse): CalibrationPriorPerturber(prior_config_61(arbitrary=arbitrary, coarse=coarse))
    for arbitrary in (False, True) for coarse in (False, True)
}

@torch.no_grad()
def make_training_window(base_window, *, generator, return_diagnostics=False):
    """Sample at most one sensor fault: Cartesian or arbitrary axis, fine/coarse, optional identity."""
    physical = physical_augmenter(base_window, generator=generator).augmented_window
    truth = physical.current_calibration
    n = next(iter(truth.values())).transform.shape[0]
    samples = {}
    for arbitrary in (False, True):
        for coarse in (False, True):
            _, xi, _ = prior_samplers_61[arbitrary, coarse](truth, generator=generator)
            samples[arbitrary, coarse] = xi
    is_cartesian = torch.rand(n, device=DEVICE, generator=generator) < CARTESIAN_PROBABILITY
    is_coarse = torch.rand(n, device=DEVICE, generator=generator) < COARSE_PRIOR_PROBABILITY
    is_identity = torch.rand(n, device=DEVICE, generator=generator) < IDENTITY_PROBABILITY
    xi = {}
    for key in CALIBRATION_KEYS:
        cart = torch.where(is_coarse[:, None], samples[False, True][key], samples[False, False][key])
        sphere = torch.where(is_coarse[:, None], samples[True, True][key], samples[True, False][key])
        xi[key] = torch.where(is_cartesian[:, None], cart, sphere).masked_fill(is_identity[:, None], 0.0)
    rotation_grid = torch.stack([xi[key][:, :3] for key in CALIBRATION_KEYS], dim=1)
    active_sensors = torch.linalg.vector_norm(rotation_grid, dim=-1) > 1e-9
    assert bool((active_sensors.sum(dim=1) <= 1).all()), 'More than one sensor fault in a sample.'
    cart_rotations = rotation_grid[is_cartesian & ~is_identity]
    assert bool(((cart_rotations.abs() > 1e-9).sum(dim=(1, 2)) <= 1).all()), 'Cartesian example has >1 nonzero axis.'
    assert all(bool((xi[key][:, 3:] == 0).all()) for key in CALIBRATION_KEYS)
    window = clone_with_current_calibration(physical, apply_independent_prior_faults(truth, xi))
    if return_diagnostics:
        return window, dict(cartesian=is_cartesian.detach().cpu().numpy(), coarse=is_coarse.detach().cpu().numpy(), identity=is_identity.detach().cpu().numpy(), active_sensor=active_sensors.detach().cpu().numpy())
    return window


In [ ]:
smoke_base = window_batch_to(collate_windows([train_dataset[i] for i in (0, 15, 60, 101)]), DEVICE, dtype=DTYPE)
smoke_window, smoke_diag = make_training_window(smoke_base, generator=augmentation_generator(TRAIN_AUG_SEED + 90001), return_diagnostics=True)
for key in CALIBRATION_KEYS:
    before = smoke_window.current_calibration[key].transform[:, :3, :3]
    target = smoke_window.targets[key].next_transform[:, :3, :3]
    assert bool(torch.isfinite(before).all() and torch.isfinite(target).all())
model.train()
smoke_out = model_predictions(smoke_window)
smoke_loss, smoke_metrics = compute_6d_objective(smoke_window, smoke_out)
smoke_loss.backward()
for part in ('transformer_encoder', 'pooler', 'mlp_body', 'head_6d'):
    grads = [p.grad for p in getattr(model, part).parameters() if p.grad is not None]
    assert grads and all(bool(torch.isfinite(g).all()) for g in grads), f'Missing or nonfinite gradients: {part}'
    print(part, 'gradient L2:', round(math.sqrt(sum(float(g.square().sum()) for g in grads)), 5))
assert bool(torch.isfinite(smoke_loss))
model.zero_grad(set_to_none=True)
model.eval()
print('PASS: full model, 6D loss, per-sample mixed augmentation and finite gradients. Loss:', round(float(smoke_loss), 5))
del smoke_out, smoke_loss, smoke_window, smoke_base


## 7. Fixed balanced validation: Cartesian X/Y/Z **and** independent arbitrary-axis directions

The probe bank is **fixed before training** and uses identical held-out physical windows and randomized true frames for every checkpoint. The Cartesian grid is retained unchanged; arbitrary-axis probes are new. The score balances both families, penalizes the worst sensor/direction task, zero-fault drift and near-179° error. Signed near-π cosines are diagnostic only, not an accuracy metric.

In [ ]:
def rotation_geodesic_deg(R_first, R_second):
    relative = R_first @ R_second.transpose(-1, -2)
    skew = torch.stack((relative[..., 2, 1] - relative[..., 1, 2], relative[..., 0, 2] - relative[..., 2, 0], relative[..., 1, 0] - relative[..., 0, 1]), dim=-1)
    sine = 0.5 * torch.linalg.vector_norm(skew, dim=-1)
    cosine = 0.5 * (relative.diagonal(dim1=-2, dim2=-1).sum(dim=-1) - 1.0)
    return torch.rad2deg(torch.atan2(sine, cosine))

def rotation_axis_cosine(R_pred, R_gt):
    """Signed SO(3) principal axis cosine, undefined for nearly zero rotation or near exact pi."""
    def skew(R):
        return torch.stack((R[..., 2, 1] - R[..., 1, 2], R[..., 0, 2] - R[..., 2, 0], R[..., 1, 0] - R[..., 0, 1]), dim=-1)
    a, b = skew(R_pred), skew(R_gt)
    an, bn = torch.linalg.vector_norm(a, dim=-1), torch.linalg.vector_norm(b, dim=-1)
    cos = (a * b).sum(-1) / (an * bn).clamp_min(1e-12)
    return torch.where((an > 1e-5) & (bn > 1e-5), cos.clamp(-1, 1), torch.full_like(cos, torch.nan))

def fixed_sphere_directions(n, seed):
    """Deterministic uniformly sampled 3D axes. Both + and - signs will be evaluated."""
    rng = np.random.default_rng(int(seed))
    u = rng.normal(size=(n, 3))
    return (u / np.linalg.norm(u, axis=1, keepdims=True)).astype(np.float64)

@torch.no_grad()
def build_probe_bank(dataset, *, n_windows, batch_size, seed, angles, n_sphere_directions):
    """List of fixed augmented windows and metadata; physical windows shared across all faults."""
    indices = np.unique(np.linspace(0, len(dataset) - 1, min(n_windows, len(dataset))).round().astype(int))
    directions_3d = fixed_sphere_directions(n_sphere_directions, seed + 700)
    axis_specs = [('cartesian', axis, tuple(np.eye(3)[j])) for j, axis in enumerate('XYZ')]
    axis_specs += [('arbitrary', f'U{j+1}', tuple(direction)) for j, direction in enumerate(directions_3d)]
    cases = []
    for group_id, start in enumerate(range(0, len(indices), batch_size)):
        chosen = indices[start:start + batch_size]
        base = window_batch_to(collate_windows([dataset[int(i)] for i in chosen]), DEVICE, dtype=DTYPE)
        physical = physical_augmenter(base, generator=augmentation_generator(seed + group_id * 1009)).augmented_window
        truth = physical.current_calibration
        n = len(chosen)
        for key in CALIBRATION_KEYS:
            # Exactly one identity probe per sensor/window, not one per direction or sign.
            if 0.0 in angles:
                cases.append(dict(fault_key=key, axis_type='identity', axis='I', axis_vector=(0., 0., 0.), sign=0, angle_deg=0.0, signed_angle_deg=0.0,
                                  window=clone_with_current_calibration(physical, truth), indices=chosen.copy(), group_id=group_id))
            for family, axis, direction in axis_specs:
                vec = torch.tensor(direction, device=DEVICE, dtype=DTYPE)
                for sign in (1, -1):
                    for angle in angles:
                        if angle == 0.0:
                            continue
                        xi = {k: torch.zeros(n, 6, device=DEVICE, dtype=DTYPE) for k in CALIBRATION_KEYS}
                        xi[key][:, :3] = sign * float(np.deg2rad(angle)) * vec[None, :]
                        prior = apply_independent_prior_faults(truth, xi)
                        cases.append(dict(fault_key=key, axis_type=family, axis=axis, axis_vector=direction, sign=sign, angle_deg=float(angle),
                                          signed_angle_deg=float(sign * angle), window=clone_with_current_calibration(physical, prior), indices=chosen.copy(), group_id=group_id))
    return cases

@torch.inference_mode()
def evaluate_probe_bank(candidate, cases, *, name):
    """One-step SO(3) error, magnitude, directional cosine, and healthy-sensor drift."""
    previous = candidate.training
    candidate.eval()
    records = []
    try:
        I = torch.eye(3, dtype=DTYPE, device=DEVICE)
        for case in cases:
            key, window = case['fault_key'], case['window']
            prior_R = window.current_calibration[key].transform[:, :3, :3]
            truth_R = window.targets[key].next_transform[:, :3, :3]
            required_R = truth_R @ prior_R.transpose(-1, -2)
            before = rotation_geodesic_deg(prior_R, truth_R)
            out = model_predictions(window, model_override=candidate).predictions
            predicted_R = out[key].delta_rotation
            after = rotation_geodesic_deg(predicted_R @ prior_R, truth_R)
            pred_mag = rotation_geodesic_deg(predicted_R, I.expand_as(predicted_R))
            cos = rotation_axis_cosine(predicted_R, required_R)
            healthy = torch.stack([rotation_geodesic_deg(out[healthy_key].delta_rotation, I.expand_as(out[healthy_key].delta_rotation)) for healthy_key in CALIBRATION_KEYS if healthy_key != key]).mean(dim=0)
            if not bool(torch.isfinite(after).all()):
                raise FloatingPointError(f'Nonfinite residual: {name}, {key}, {case["axis"]}')
            if case['angle_deg'] > 0 and not torch.allclose(before, torch.full_like(before, case['angle_deg']), atol=.08, rtol=0):
                raise AssertionError(f'Fault is not the requested angle: {key}, {case["axis"]}, {case["angle_deg"]}')
            for j, index in enumerate(case['indices']):
                b, a = float(before[j]), float(after[j])
                records.append(dict(model=name, fault_key=key, axis_type=case['axis_type'], axis=case['axis'], sign=case['sign'],
                                    angle_deg=case['angle_deg'], signed_angle_deg=case['signed_angle_deg'], physical_index=int(index),
                                    before_deg=b, after_deg=a, relative_remaining=a/b if b > 1e-3 else np.nan,
                                    correction_deg=float(pred_mag[j]), axis_cosine=float(cos[j]), healthy_predicted_drift_deg=float(healthy[j])))
    finally:
        candidate.train(previous)
    return pd.DataFrame.from_records(records)

def probe_metrics(frame):
    use = frame.loc[frame.angle_deg.isin(PROBE_SCORE_ANGLES_DEG) & (frame.angle_deg > 0)]
    group = use.groupby(['fault_key', 'axis_type', 'axis'], as_index=False).agg(relative_remaining=('relative_remaining', 'mean'), mean_after_deg=('after_deg', 'mean'))
    cartesian = group.loc[group.axis_type == 'cartesian']
    arbitrary = group.loc[group.axis_type == 'arbitrary']
    if len(cartesian) != 9 or len(arbitrary) == 0:
        raise RuntimeError(f'Incomplete probe bank: cartesian={len(cartesian)}/9, arbitrary={len(arbitrary)}')
    cart_mean, arb_mean = float(cartesian.relative_remaining.mean()), float(arbitrary.relative_remaining.mean())
    worst = float(group.relative_remaining.max())
    base_mean = CARTESIAN_PROBABILITY * cart_mean + (1 - CARTESIAN_PROBABILITY) * arb_mean
    near_pi = float(use.loc[use.angle_deg == 179.0, 'relative_remaining'].mean())
    zero = float(frame.loc[frame.angle_deg == 0.0, 'after_deg'].mean())
    score = base_mean + WORST_PAIR_WEIGHT * max(0.0, worst - base_mean) + NEAR_PI_WEIGHT * near_pi + ZERO_DRIFT_WEIGHT * zero / 30.0
    metrics = dict(score=score, cartesian_mean=cart_mean, arbitrary_mean=arb_mean, worst_task=worst, near_pi_ratio=near_pi, zero_drift_deg=zero)
    return metrics, group

print('Defined paired Cartesian + arbitrary-axis probes, SO(3) residual and cosine metrics.')


## 8. Reversible stochastic preflight (mixed axes, no full training state changed)

Starts from **002.30**, tests the actual mixed sampler, runs an independent AdamW optimizer, evaluates fixed Cartesian and arbitrary-axis probes, saves its best and final candidate, then restores original model/RNG states. The full run **does not** automatically initialize from this preflight candidate.

In [ ]:
PREFLIGHT_DIR = RUN_DIR / 'stochastic_mixed_axes'
PREFLIGHT_BEST_PATH = PREFLIGHT_DIR / 'best_state.pt'
PREFLIGHT_LAST_PATH = PREFLIGHT_DIR / 'last_state.pt'
PREFLIGHT_DIR.mkdir(parents=True, exist_ok=True)
preflight_history = []

if RUN_STOCHASTIC_PREFLIGHT:
    pre_cases = build_probe_bank(val_dataset, n_windows=PREFLIGHT_PROBE_WINDOWS, batch_size=PREFLIGHT_PROBE_BATCH_SIZE,
                                 seed=PROBE_SEED + 111, angles=(0., 30., 90., 150., 175., 179.), n_sphere_directions=PREFLIGHT_3D_DIRECTIONS)
    source_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
    source_training, source_cpu_rng = model.training, torch.get_rng_state().clone()
    source_cuda_rng = torch.cuda.get_rng_state_all() if DEVICE.type == 'cuda' else None
    source_numpy_rng, source_python_rng = np.random.get_state(), random.getstate()
    pre_optimizer = torch.optim.AdamW(parameter_groups(model, transformer_lr=PREFLIGHT_TRANSFORMER_LR, other_lr=PREFLIGHT_HEAD_LR), weight_decay=WEIGHT_DECAY)
    pre_gen = augmentation_generator(TRAIN_AUG_SEED + 6_100_000)
    index_rng = np.random.default_rng(TRAIN_AUG_SEED + 6_100_001)
    best_pre_score = float('inf')
    try:
        for step in range(PREFLIGHT_STEPS + 1):
            if step > 0:
                model.train()
                pre_optimizer.zero_grad(set_to_none=True)
                indices = index_rng.choice(len(train_dataset), size=min(PREFLIGHT_BATCH_SIZE, len(train_dataset)), replace=False)
                base = window_batch_to(collate_windows([train_dataset[int(i)] for i in indices]), DEVICE, dtype=DTYPE)
                batch = make_training_window(base, generator=pre_gen)
                output = model_predictions(batch)
                loss, _ = compute_6d_objective(batch, output)
                if not torch.isfinite(loss):
                    raise FloatingPointError(f'Nonfinite preflight loss at step {step}')
                loss.backward()
                norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
                if not torch.isfinite(norm):
                    raise FloatingPointError(f'Nonfinite gradient at step {step}')
                pre_optimizer.step()
                train_loss, grad_norm = float(loss.detach()), float(norm)
                del base, batch, output, loss
            else:
                train_loss, grad_norm = float('nan'), float('nan')
            if step == 0 or step % PREFLIGHT_REPORT_EVERY == 0 or step == PREFLIGHT_STEPS:
                frame = evaluate_probe_bank(model, pre_cases, name='preflight')
                metrics, pairs = probe_metrics(frame)
                entry = dict(step=step, train_loss=train_loss, grad_norm=grad_norm, **metrics)
                preflight_history.append(entry)
                print(f'preflight {step:4d}/{PREFLIGHT_STEPS} | train={train_loss:.4f} | score={metrics["score"]:.4f} | '
                      f'cartesian={metrics["cartesian_mean"]:.3f}, arbitrary={metrics["arbitrary_mean"]:.3f}, '
                      f'worst={metrics["worst_task"]:.3f}, near179={metrics["near_pi_ratio"]:.3f}', flush=True)
                if step > 0 and metrics['score'] < best_pre_score:
                    best_pre_score = metrics['score']
                    torch.save({'model_state_dict': {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, 'step': step, 'probe_metrics': metrics}, PREFLIGHT_BEST_PATH)
                if step == PREFLIGHT_STEPS:
                    torch.save({'model_state_dict': {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, 'step': step, 'probe_metrics': metrics}, PREFLIGHT_LAST_PATH)
    finally:
        model.load_state_dict(source_state, strict=True)
        model.train(source_training)
        model.zero_grad(set_to_none=True)
        torch.set_rng_state(source_cpu_rng)
        if source_cuda_rng is not None:
            torch.cuda.set_rng_state_all(source_cuda_rng)
        np.random.set_state(source_numpy_rng)
        random.setstate(source_python_rng)
        del source_state, pre_optimizer
        gc.collect()
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()
        print('Restored exact 002.30 initialization and RNG; stochastic optimizer discarded.')
    trace = pd.DataFrame(preflight_history)
    trace.to_csv(PREFLIGHT_DIR / 'history.csv', index=False)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(trace.step, trace.score, '-o', label='Balanced score')
    axes[0].plot(trace.step, trace.worst_task, '--', label='Worst sensor/direction')
    axes[1].plot(trace.step, trace.cartesian_mean, '-o', label='Cartesian X/Y/Z')
    axes[1].plot(trace.step, trace.arbitrary_mean, '-o', label='Arbitrary 3D axes')
    axes[1].plot(trace.step, trace.near_pi_ratio, '--', label='179° relative error')
    for ax in axes:
        ax.set(xlabel='Optimizer updates', ylabel='Relative remaining SO(3) error / score')
        ax.grid(alpha=.25)
        ax.legend()
    fig.suptitle('Notebook 61: reversible stochastic fine-tuning from 002.30')
    fig.tight_layout()
    fig.savefig(PREFLIGHT_DIR / 'learning_curves.png', dpi=170)
    if SHOW_PLOTS:
        plt.show()
    else:
        plt.close(fig)
else:
    print('Stochastic preflight disabled; run the full setup cell next.')


## 9. Full training and exact Notebook 61 resume

**New run:** initialize all 002.30 weights, create a new AdamW and scheduler. **Resume:** set `RESUME_FULL_TRAINING=True` and increase `FULL_TARGET_EPOCHS`; load **61 `last.pt`**, restoring the full model, AdamW moments, scheduler, both generators, epoch, global step and histories. `best.pt` remains protected until the balanced probe score improves. The learning-rate scheduler monitors the **same fixed probe score** as checkpointing; a probe occurs every `PROBE_EVERY_EPOCHS` epochs. This differs intentionally from Notebook 60 (validation loss), and is part of the saved experiment configuration.

In [ ]:
def run_6d_epoch(loader, *, training, generator, max_batches=None):
    model.train(training)
    sums, counts = defaultdict(float), defaultdict(float)
    n_samples, optimizer_updates = 0, 0
    pending_samples, pending_microbatches = 0, 0
    started = time.perf_counter()
    if training:
        optimizer.zero_grad(set_to_none=True)

    def optimizer_flush():
        nonlocal pending_samples, pending_microbatches, optimizer_updates
        if pending_samples == 0:
            return
        for p in model.parameters():
            if p.grad is not None:
                p.grad.div_(pending_samples)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        optimizer_updates += 1
        pending_samples, pending_microbatches = 0, 0

    with (torch.enable_grad() if training else torch.no_grad()):
        for batch_id, cpu_batch in enumerate(loader):
            if max_batches is not None and batch_id >= max_batches:
                break
            base = window_batch_to(cpu_batch, DEVICE, dtype=DTYPE)
            augmented = make_training_window(base, generator=generator)
            prediction = model_predictions(augmented)
            loss, metrics = compute_6d_objective(augmented, prediction)
            n = len(next(iter(augmented.current_calibration.values())).transform)
            if training:
                (loss * n).backward()
                pending_samples += n
                pending_microbatches += 1
                if pending_microbatches >= GRADIENT_ACCUMULATION_STEPS:
                    optimizer_flush()
            for key, value in metrics.items():
                if key == 'active_rotation_count':
                    continue
                sums[key] += float(value) * n
                counts[key] += n
            n_samples += n
            del base, augmented, prediction, loss
        if training:
            optimizer_flush()
    if not n_samples:
        raise RuntimeError('Epoch used no samples')
    result = metric_average(sums, counts)
    result.update(n_samples=n_samples, optimizer_updates=optimizer_updates, elapsed_s=time.perf_counter()-started)

    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    return result


In [ ]:
BEST_CHECKPOINT_PATH = RUN_DIR / 'best.pt'
LAST_CHECKPOINT_PATH = RUN_DIR / 'last.pt'
HISTORY_PATH = RUN_DIR / 'training_6d_history.csv'
PAIR_HISTORY_PATH = RUN_DIR / 'probe_sensor_direction_history.csv'
PREFLIGHT_INIT_PATH = PREFLIGHT_BEST_PATH
FULL_INIT_PATH = PREFLIGHT_INIT_PATH if FULL_INIT_FROM_PREFLIGHT_BEST else SOURCE_230_PATH

full_probe_bank = build_probe_bank(val_dataset, n_windows=FULL_PROBE_WINDOWS, batch_size=FULL_PROBE_BATCH_SIZE,
                                  seed=PROBE_SEED, angles=PROBE_ANGLES_DEG, n_sphere_directions=FULL_PROBE_3D_DIRECTIONS)
optimizer = torch.optim.AdamW(parameter_groups(model), weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=LR_REDUCE_FACTOR, patience=LR_PATIENCE_PROBES)
train_generator = augmentation_generator(TRAIN_AUG_SEED + 61_000)
start_epoch, global_step, best_score, best_epoch = 0, 0, float('inf'), 0
history, pair_history = [], []

checkpoint_config = dict(
    run_name=RUN_NAME, source_checkpoint=str(SOURCE_230_PATH), full_init_checkpoint=str(FULL_INIT_PATH),
    architecture='PooledCalibration6DModel', calibration_keys=tuple(CALIBRATION_KEYS),
    train_names=tuple(TRAIN_NAMES), val_names=tuple(VAL_NAMES),
    fine_max_deg=FINE_ROTATION_MAX_DEG, fine_scale_deg=FINE_ROTATION_SCALE_DEG,
    coarse_range_deg=(COARSE_ROTATION_MIN_DEG, COARSE_ROTATION_MAX_DEG), coarse_probability=COARSE_PRIOR_PROBABILITY,
    cartesian_probability=CARTESIAN_PROBABILITY, identity_probability=IDENTITY_PROBABILITY,
    one_faulted_sensor=True, arbitrary_axis_isotropic=True,
    randomize_true_sensor_frames=RANDOMIZE_TRUE_SENSOR_FRAMES, true_rotation_max_rad=TRUE_CALIBRATION_ROTATION_MAX_RAD,
    loss_weights=asdict(LOSS_WEIGHTS), active_weight=ACTIVE_ROTATION_SUPERVISION_WEIGHT,
    train_batch_size=TRAIN_BATCH_SIZE, gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    transformer_lr=TRANSFORMER_LR, head_mlp_lr=HEAD_MLP_LR, weight_decay=WEIGHT_DECAY,
    scheduler='ReduceLROnPlateau(probe_score)', lr_reduce_factor=LR_REDUCE_FACTOR, lr_patience_probes=LR_PATIENCE_PROBES,
    probe_angles=PROBE_ANGLES_DEG, probe_score_angles=PROBE_SCORE_ANGLES_DEG, probe_seed=PROBE_SEED,
    probe_windows=FULL_PROBE_WINDOWS, probe_3d_directions=FULL_PROBE_3D_DIRECTIONS,
    probe_every_epochs=PROBE_EVERY_EPOCHS, worst_pair_weight=WORST_PAIR_WEIGHT,
    near_pi_weight=NEAR_PI_WEIGHT, zero_drift_weight=ZERO_DRIFT_WEIGHT,
    tau_reference_s=dict(REFINED_TAU_BY_SENSOR_S),
)

if RUN_FULL_TRAINING:
    if RESUME_FULL_TRAINING:
        if not LAST_CHECKPOINT_PATH.is_file():
            raise FileNotFoundError(f'No Notebook 61 last.pt for resume: {LAST_CHECKPOINT_PATH}')
        # Check exact saved experiment settings before restoring optimizer/scheduler.
        raw = torch.load(LAST_CHECKPOINT_PATH, map_location='cpu', weights_only=False)  # Only your trusted local checkpoint.
        saved_extra = raw.get('extra_state') or {}
        saved_cfg = saved_extra.get('config_6d')
        if saved_cfg != checkpoint_config:
            differences = {k: (saved_cfg.get(k, '<missing>') if saved_cfg else '<missing>', checkpoint_config.get(k, '<missing>'))
                           for k in (set(saved_cfg or {}) | set(checkpoint_config))
                           if (saved_cfg or {}).get(k, '<missing>') != checkpoint_config.get(k, '<missing>')}
            raise RuntimeError(f'Resume configuration changed; use a NEW run name: {differences}')
        best_score = float(saved_extra['best_probe_score'])
        best_epoch = int(saved_extra['best_epoch'])
        del raw
        loaded = load_training_checkpoint(LAST_CHECKPOINT_PATH, model=model, optimizer=optimizer, scheduler=scheduler,
                                          generators={'train_shuffle': train_shuffle_generator, 'train_aug': train_generator},
                                          map_location=DEVICE, restore_rng_state=True)
        start_epoch, global_step = int(loaded['epoch']), int(loaded['global_step'])
        if not optimizer.state:
            raise RuntimeError('Empty AdamW state after resume. The optimizer was NOT really restored.')
        if not HISTORY_PATH.is_file():
            raise FileNotFoundError(f'Cannot safely resume without history CSV: {HISTORY_PATH}')
        frame = pd.read_csv(HISTORY_PATH)
        frame = frame.loc[frame.epoch <= start_epoch]
        if frame.empty or int(frame.epoch.iloc[-1]) != start_epoch:
            raise RuntimeError(f'Training history missing completed epoch {start_epoch}.')
        history = frame.to_dict('records')
        if PAIR_HISTORY_PATH.is_file():
            pair_frame = pd.read_csv(PAIR_HISTORY_PATH)
            pair_history = pair_frame.loc[pair_frame.epoch <= start_epoch].to_dict('records')
        if FULL_TARGET_EPOCHS <= start_epoch:
            raise ValueError(f'Target epoch {FULL_TARGET_EPOCHS} must be > completed epoch {start_epoch}')
        del frame, loaded
        print(f'RESUMED Notebook 61: {start_epoch} completed epochs, target={FULL_TARGET_EPOCHS}, global_step={global_step}')
        print('Previous best:', best_epoch, best_score, '| AdamW state entries:', len(optimizer.state))
    else:
        if any(p.is_file() for p in (BEST_CHECKPOINT_PATH, LAST_CHECKPOINT_PATH, HISTORY_PATH)):
            raise FileExistsError(f'{RUN_DIR} already contains checkpoints; set RESUME_FULL_TRAINING=True or choose a new RUN_NAME.')
        if FULL_INIT_FROM_PREFLIGHT_BEST:
            if not FULL_INIT_PATH.is_file():
                raise FileNotFoundError(FULL_INIT_PATH)
            state = torch.load(FULL_INIT_PATH, map_location='cpu', weights_only=True)
            model.load_state_dict(state['model_state_dict'], strict=True)
            del state
        else:
            load_training_checkpoint(SOURCE_230_PATH, model=model, map_location=DEVICE, restore_rng_state=False)
        print('NEW Notebook 61 training: full model initialized from', FULL_INIT_PATH)
        print('AdamW is new; NO optimizer state is reused from 002.30.')
        initial_frame = evaluate_probe_bank(model, full_probe_bank, name='61_initial')
        initial_metrics, initial_pairs = probe_metrics(initial_frame)
        print('Initial balanced probe:', {k: round(v, 4) for k, v in initial_metrics.items()})
else:
    print('Full training disabled. Set RUN_FULL_TRAINING=True to start or resume training.')


def save_61_checkpoint(path, epoch_number, metrics):
    save_training_checkpoint(path, model=model, optimizer=optimizer, scheduler=scheduler, epoch=epoch_number, global_step=global_step,
                             metrics=metrics, generators={'train_shuffle': train_shuffle_generator, 'train_aug': train_generator},
                             extra_state={'config_6d': checkpoint_config, 'best_probe_score': best_score, 'best_epoch': best_epoch})


In [ ]:
if RUN_FULL_TRAINING:
    for epoch in range(start_epoch, FULL_TARGET_EPOCHS):
        train_shuffle_generator.manual_seed(RANDOM_SEED + 12001 + epoch)
        train = run_6d_epoch(train_loader, training=True, generator=train_generator, max_batches=MAX_TRAIN_BATCHES_PER_EPOCH)
        validation = run_6d_epoch(val_loader, training=False, generator=augmentation_generator(VAL_AUG_SEED + 6100), max_batches=MAX_VAL_BATCHES_PER_EPOCH)
        global_step += train['optimizer_updates']
        improved = False
        row = dict(epoch=epoch + 1, global_step=global_step, train_loss=train['loss_total'], val_loss=validation['loss_total'],
                   train_rotation_6d=train['loss_rotation_6d'], val_rotation_6d=validation['loss_rotation_6d'],
                   train_active_deg=train['active_rotation_error_deg'], val_active_deg=validation['active_rotation_error_deg'])
        if (epoch + 1) % PROBE_EVERY_EPOCHS == 0:
            probe = evaluate_probe_bank(model, full_probe_bank, name='61_train_probe')
            metrics, pairs = probe_metrics(probe)
            row.update({f'probe_{k}': v for k, v in metrics.items()})
            scheduler.step(metrics['score'])  # Same fixed validation objective as checkpoint selection.
            pairs['epoch'], pairs['global_step'] = epoch + 1, global_step
            pair_history.extend(pairs.to_dict('records'))
            if metrics['score'] < best_score:
                best_score, best_epoch = metrics['score'], epoch + 1
                improved = True
        row.update(head_mlp_lr=optimizer.param_groups[1]['lr'], transformer_lr=optimizer.param_groups[0]['lr'])
        history.append(row)
        # Save optimizer/scheduler AND augmentation/shuffle generator states every epoch.
        if improved:
            save_61_checkpoint(BEST_CHECKPOINT_PATH, epoch + 1, row)
        pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)
        if pair_history:
            pd.DataFrame(pair_history).to_csv(PAIR_HISTORY_PATH, index=False)
        save_61_checkpoint(LAST_CHECKPOINT_PATH, epoch + 1, row)
        score_text = f'probe={row["probe_score"]:.4f} cart={row["probe_cartesian_mean"]:.3f} 3d={row["probe_arbitrary_mean"]:.3f} near179={row["probe_near_pi_ratio"]:.3f}' if 'probe_score' in row else 'probe not evaluated'
        print(f'Epoch {epoch+1:3d}/{FULL_TARGET_EPOCHS} train={train["loss_total"]:.4f} val={validation["loss_total"]:.4f} | {score_text} | '
              f'LR={row["transformer_lr"]:.1e}/{row["head_mlp_lr"]:.1e}'+(' BEST' if improved else ''), flush=True)
    print('Full training complete. Best epoch/score:', best_epoch, best_score)
else:
    print('No full training requested. Existing Notebook 61 checkpoints, if any, remain untouched.')


In [ ]:
if HISTORY_PATH.is_file():
    history_view = pd.read_csv(HISTORY_PATH)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
    axes[0].plot(history_view.epoch, history_view.train_loss, label='Train')
    axes[0].plot(history_view.epoch, history_view.val_loss, label='Validation')
    axes[0].set(title='Supervised loss', xlabel='Epoch', ylabel='6D objective')
    if 'probe_score' in history_view:
        axes[1].plot(history_view.epoch, history_view.probe_score, '-o', label='Balanced score')
        axes[1].plot(history_view.epoch, history_view.probe_cartesian_mean, label='Cartesian')
        axes[1].plot(history_view.epoch, history_view.probe_arbitrary_mean, label='Arbitrary 3D')
        axes[1].plot(history_view.epoch, history_view.probe_near_pi_ratio, '--', label='179° relative')
    axes[1].set(title='Fixed angular validation', xlabel='Epoch', ylabel='Relative SO(3) error')
    axes[2].plot(history_view.epoch, history_view.transformer_lr, label='Transformer')
    axes[2].plot(history_view.epoch, history_view.head_mlp_lr, label='Pooler/MLP/6D head')
    axes[2].set(title='Restorable optimizer learning rates', xlabel='Epoch', ylabel='Learning rate', yscale='log')
    for ax in axes:
        ax.grid(alpha=.25)
        ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(RUN_DIR / '61_training_overview.png', dpi=175)
    if SHOW_PLOTS:
        plt.show()
    else:
        plt.close(fig)
    print('Saved full-training history:', HISTORY_PATH)
else:
    print('No full-run history yet.')


## 10. Post-training paired one-step comparison: 002.30 source vs 61 best

Set `EVAL_RUN=True` in settings after at least one full-run probe saves `best.pt`. These cells load **separate frozen complete models**. Checkpoint evaluation cases have a different seed from checkpoint selection and reuse exactly the same physical windows, GT rig, perturbation directions, signs and magnitudes. `EVAL_INCLUDE_227=True` optionally adds the original 002.27 model as a third reference.

In [ ]:
EVAL_DIR = RUN_DIR / 'plots' / 'post_training_mixed_axes_comparison'
if EVAL_RUN:
    EVAL_DIR.mkdir(parents=True, exist_ok=True)
    if not BEST_CHECKPOINT_PATH.is_file():
        raise FileNotFoundError(f'Full-run best checkpoint missing: {BEST_CHECKPOINT_PATH}')
    if EVAL_INCLUDE_227 and not SOURCE_227_PATH.is_file():
        raise FileNotFoundError(f'Optional 002.27 checkpoint missing: {SOURCE_227_PATH}')

    def load_frozen_6d(path):
        candidate = PooledCalibration6DModel(model_config).to(device=DEVICE, dtype=DTYPE)
        load_training_checkpoint(path, model=candidate, map_location=DEVICE, restore_rng_state=False)
        return candidate.eval().requires_grad_(False)

    eval_models = {'2.30 source': load_frozen_6d(SOURCE_230_PATH), '61 best': load_frozen_6d(BEST_CHECKPOINT_PATH)}
    if EVAL_INCLUDE_227:
        eval_models['2.27 original'] = load_frozen_6d(SOURCE_227_PATH)
    eval_cases = build_probe_bank(val_dataset, n_windows=EVAL_WINDOWS, batch_size=EVAL_BATCH_SIZE, seed=VAL_AUG_SEED + 6_110_101,
                                  angles=EVAL_ANGLES_DEG, n_sphere_directions=EVAL_3D_DIRECTIONS)
    print('Complete checkpoints:', SOURCE_230_PATH, BEST_CHECKPOINT_PATH)
    print('Evaluation case count:', len(eval_cases), '| each checkpoint gets exactly the same cases')
    print('Model parameter delta L2:', math.sqrt(sum(float((a - b).square().sum()) for a, b in zip(eval_models['2.30 source'].parameters(), eval_models['61 best'].parameters()))))
else:
    print('Set EVAL_RUN=True after training to load 002.30 and 61 best for paired validation.')


In [ ]:
if EVAL_RUN:
    one_frames = []
    for label, candidate in eval_models.items():
        print('Paired one-step evaluation:', label, flush=True)
        one_frames.append(evaluate_probe_bank(candidate, eval_cases, name=label))
    one_df = pd.concat(one_frames, ignore_index=True)
    one_df['axis_group'] = np.where(one_df.axis_type == 'cartesian', one_df.axis, np.where(one_df.axis_type == 'arbitrary', '3D', 'I'))
    one_df.to_csv(EVAL_DIR / 'one_step_individual.csv', index=False)
    one_summary = one_df.groupby(['model', 'fault_key', 'axis_type', 'axis_group', 'axis', 'angle_deg', 'signed_angle_deg'], as_index=False).agg(
        mean_after_deg=('after_deg', 'mean'), median_after_deg=('after_deg', 'median'), p90_after_deg=('after_deg', lambda x: x.quantile(.90)),
        mean_correction_deg=('correction_deg', 'mean'), mean_axis_cosine=('axis_cosine', 'mean'),
        success_10deg=('after_deg', lambda x: (x < 10).mean()), success_30deg=('after_deg', lambda x: (x < 30).mean()), n=('after_deg', 'size'))
    one_summary.to_csv(EVAL_DIR / 'one_step_summary_signed.csv', index=False)
    pair_keys = ['fault_key', 'axis_type', 'axis', 'signed_angle_deg', 'physical_index']
    paired = one_df[one_df.model.isin(('2.30 source', '61 best'))].pivot(index=pair_keys, columns='model', values='after_deg').dropna()
    assert len(paired) * 2 == len(one_df[one_df.model.isin(('2.30 source', '61 best'))]), 'One-step cases are not properly paired.'
    paired['improvement_deg'] = paired['2.30 source'] - paired['61 best']
    paired.reset_index().to_csv(EVAL_DIR / 'one_step_paired_improvement.csv', index=False)
    active = one_df[one_df.angle_deg > 0]
    print('Mean residual ratio by sensor/axis family (lower is better):')
    display(active.groupby(['model', 'fault_key', 'axis_group']).relative_remaining.mean().unstack('axis_group').round(3))
    print('Mean SO(3) residual at ±179° by sensor/axis family (degrees):')
    display(active[active.angle_deg == 179.].groupby(['model', 'fault_key', 'axis_group']).after_deg.mean().unstack('axis_group').round(2))
    print('Paired improvement in degrees across angles: positive means 61 best is better.')
    display(paired.groupby(level=['fault_key', 'axis_type', 'axis']).improvement_deg.mean().unstack('axis').round(2))
else:
    print('One-step comparison disabled.')


In [ ]:
if EVAL_RUN:
    color = {'2.30 source': 'tab:blue', '61 best': 'tab:orange', '2.27 original': 'tab:green'}

    def plot_four_column(metric, title, y_label, filename, *, reference=False, ylim=None):
        fig, axes = plt.subplots(3, 4, figsize=(19, 11), sharex=True, sharey=True)
        for row, key in enumerate(CALIBRATION_KEYS):
            for col, direction in enumerate(('X', 'Y', 'Z', '3D')):
                ax = axes[row, col]
                subset = one_df[(one_df.fault_key == key) & (one_df.axis_group == direction) & (one_df.angle_deg > 0)]
                curves = subset.groupby(['model', 'sign', 'angle_deg'], as_index=False).agg(value=(metric, 'mean'))
                for model_name in eval_models:
                    for sign, style in ((1, '-'), (-1, '--')):
                        line = curves[(curves.model == model_name) & (curves.sign == sign)].sort_values('angle_deg')
                        ax.plot(line.angle_deg, line.value, style, marker='.', color=color[model_name], label=f'{model_name} {"+" if sign == 1 else "−"}')
                if reference:
                    ax.plot(EVAL_ANGLES_DEG, EVAL_ANGLES_DEG, ':', color='gray', linewidth=1.2, label='No correction / GT magnitude')
                if ylim is not None:
                    ax.set_ylim(*ylim)
                ax.set_title(f'{key} / {direction}')
                ax.grid(alpha=.25)
                if row == 2:
                    ax.set_xlabel('Initial error magnitude (°)')
                if col == 0:
                    ax.set_ylabel(y_label)
        axes[0, 0].legend(fontsize=7, ncol=2)
        fig.suptitle(title, fontsize=15)
        fig.tight_layout()
        fig.savefig(EVAL_DIR / filename, dpi=175)
        if SHOW_PLOTS:
            plt.show()
        else:
            plt.close(fig)

    plot_four_column('after_deg', 'One-window SO(3) correction: Cartesian X/Y/Z and arbitrary 3D', 'Mean remaining error (°)', 'one_step_residuals.png', reference=True, ylim=(0, 180))
    plot_four_column('correction_deg', 'Predicted correction magnitude vs target magnitude', 'Mean predicted step (°)', 'one_step_correction_magnitude.png', reference=True, ylim=(0, 180))
    plot_four_column('axis_cosine', 'Predicted-vs-GT principal axis cosine (caution near 180°)', 'Mean signed axis cosine', 'one_step_axis_cosine.png', ylim=(-1.05, 1.05))

    # Inspect 179° distributions rather than only signed means; opposite axes may be close on SO(3).
    fig, axes = plt.subplots(3, 4, figsize=(18, 10), sharex=True, sharey=True)
    for row, key in enumerate(CALIBRATION_KEYS):
        for col, direction in enumerate(('X', 'Y', 'Z', '3D')):
            ax = axes[row, col]
            values = one_df[(one_df.fault_key == key) & (one_df.axis_group == direction) & (one_df.angle_deg == 179.0)]
            for label in eval_models:
                arr = values.loc[values.model == label, 'axis_cosine'].dropna().to_numpy()
                ax.hist(arr, bins=np.linspace(-1, 1, 17), histtype='step', linewidth=1.6, color=color[label], label=f'{label} (n={len(arr)})')
            ax.axvline(0, linestyle=':', color='gray')
            ax.set_title(f'{key} / {direction}')
            ax.grid(alpha=.2)
            if row == 2:
                ax.set_xlabel('Axis cosine at 179°')
            if col == 0:
                ax.set_ylabel('Number of cases')
    axes[0, 0].legend(fontsize=7)
    fig.suptitle('Per-case 179° correction-direction distributions')
    fig.tight_layout()
    fig.savefig(EVAL_DIR / 'one_step_179_axis_cosine_histograms.png', dpi=175)
    if SHOW_PLOTS:
        plt.show()
    else:
        plt.close(fig)

    fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.8))
    values = one_df[one_df.angle_deg == 179.0]
    heatmaps = {label: values[values.model == label].groupby(['fault_key', 'axis_group']).after_deg.mean().unstack('axis_group').reindex(index=CALIBRATION_KEYS, columns=('X', 'Y', 'Z', '3D')) for label in ('2.30 source', '61 best')}
    heatmaps['Improvement'] = heatmaps['2.30 source'] - heatmaps['61 best']
    for ax, (label, matrix) in zip(axes, heatmaps.items()):
        val = matrix.to_numpy()
        limit = max(15, float(np.nanmax(np.abs(val))))
        pic = ax.imshow(val, cmap='RdBu' if label == 'Improvement' else 'viridis_r', vmin=-limit if label == 'Improvement' else 0, vmax=limit if label == 'Improvement' else 180)
        ax.set_xticks(range(4), ('X', 'Y', 'Z', '3D'))
        ax.set_yticks(range(3), CALIBRATION_KEYS)
        ax.set_title(label)
        for i in range(3):
            for j in range(4):
                ax.text(j, i, f'{matrix.iloc[i, j]:+.1f}' if label == 'Improvement' else f'{matrix.iloc[i, j]:.1f}', ha='center', va='center', fontsize=9)
        fig.colorbar(pic, ax=ax, shrink=.76)
    fig.suptitle('One-step remaining error at ±179° and paired improvement (degrees)')
    fig.tight_layout()
    fig.savefig(EVAL_DIR / 'one_step_179_heatmaps.png', dpi=175)
    if SHOW_PLOTS:
        plt.show()
    else:
        plt.close(fig)
else:
    print('One-step plotting disabled.')


## 11. Matched 12-window chronological rollouts: Cartesian and arbitrary 3D axes

Both models receive exactly the same initial prior, physical motion windows and true rig. Each feeds its **own** predicted rotation back into the next five-second window. Only the intentionally faulted sensor rotation changes; translation and time offsets are preserved. The 3D-axis rollouts use fixed independent isotropic axes. All results, success rates and per-case cosines are saved.

In [ ]:
@torch.inference_mode()
def evaluate_12_window_rollouts(models, *, sequences=EVAL_ROLLOUT_SEQUENCES, starts=EVAL_ROLLOUT_STARTS,
                                truths=EVAL_ROLLOUT_TRUTHS, steps=EVAL_ROLLOUT_STEPS, angles=EVAL_ROLLOUT_ANGLES,
                                sphere_directions=EVAL_ROLLOUT_3D_DIRECTIONS, seed=VAL_AUG_SEED + 6_110_303):
    """Independent model feedback, matched GT and per-window raw measurements; only faulted R updates."""
    direction_specs = [('cartesian', axis, tuple(np.eye(3)[j])) for j, axis in enumerate('XYZ')]
    direction_specs += [('arbitrary', f'U{j+1}', tuple(vec)) for j, vec in enumerate(fixed_sphere_directions(sphere_directions, seed + 123))]
    direction_specs += [('identity', 'I', (0., 0., 0.))]
    I = torch.eye(3, device=DEVICE, dtype=DTYPE)
    records = []
    for seq_id, sequence in enumerate(sequences):
        sequence_indices = np.asarray(val_dataset.window_indices_for_sequence(sequence), dtype=np.int64)
        last_start = len(sequence_indices) - steps
        if last_start < 0:
            raise ValueError(f'{sequence}: expected {steps} consecutive windows, found {len(sequence_indices)}')
        start_positions = np.unique(np.linspace(0, last_start, min(starts, last_start + 1)).round().astype(int))
        for start_id, start in enumerate(start_positions):
            window_indices = sequence_indices[start:start + steps]
            for truth_id in range(truths):
                truth_seed = seed + seq_id * 1_000_003 + start_id * 100_003 + truth_id * 1009
                physical_windows = []
                for index in window_indices:
                    base = window_batch_to(collate_windows([val_dataset[int(index)]]), DEVICE, dtype=DTYPE)
                    # Reset to the same seed for each window so GT is constant across time.
                    physical = physical_augmenter(base, generator=augmentation_generator(truth_seed)).augmented_window
                    physical_windows.append(physical)
                truth = physical_windows[0].current_calibration
                for physical in physical_windows[1:]:
                    for key in CALIBRATION_KEYS:
                        torch.testing.assert_close(physical.current_calibration[key].transform, truth[key].transform, atol=2e-5, rtol=2e-5)
                        torch.testing.assert_close(physical.current_calibration[key].time_offset, truth[key].time_offset, atol=2e-6, rtol=2e-6)
                for fault_key in CALIBRATION_KEYS:
                    for family, axis, direction in direction_specs:
                        direction_vec = torch.tensor(direction, device=DEVICE, dtype=DTYPE)
                        these_angles = (0.,) if family == 'identity' else tuple(a for a in angles if a > 0)
                        for initial_angle in these_angles:
                            for sign in ((0,) if initial_angle == 0 else (1, -1)):
                                xi = {key: torch.zeros(1, 6, device=DEVICE, dtype=DTYPE) for key in CALIBRATION_KEYS}
                                xi[fault_key][:, :3] = float(np.deg2rad(initial_angle)) * sign * direction_vec[None, :]
                                initial_states = apply_independent_prior_faults(truth, xi)
                                for model_label, candidate in models.items():
                                    states = {key: replace(state, transform=state.transform.clone(), time_offset=state.time_offset.clone()) for key, state in initial_states.items()}
                                    for step in range(steps + 1):
                                        prior_R = states[fault_key].transform[:, :3, :3]
                                        truth_R = truth[fault_key].transform[:, :3, :3]
                                        residual = float(rotation_geodesic_deg(prior_R, truth_R)[0])
                                        record = dict(model=model_label, sequence=sequence, start_index=int(window_indices[0]), truth_id=truth_id,
                                                      fault_key=fault_key, axis_type=family, axis=axis, axis_group=axis if family == 'cartesian' else ('3D' if family == 'arbitrary' else 'I'),
                                                      sign=int(sign), initial_angle_deg=float(initial_angle), signed_angle_deg=float(sign * initial_angle),
                                                      step=step, residual_deg=residual, predicted_step_deg=np.nan, axis_cosine=np.nan)
                                        if step < steps:
                                            input_window = clone_with_current_calibration(physical_windows[step], states)
                                            predicted_R = model_predictions(input_window, model_override=candidate).predictions[fault_key].delta_rotation
                                            desired_R = truth_R @ prior_R.transpose(-1, -2)
                                            record['predicted_step_deg'] = float(rotation_geodesic_deg(predicted_R, I.expand_as(predicted_R))[0])
                                            record['axis_cosine'] = float(rotation_axis_cosine(predicted_R, desired_R)[0])
                                            next_T = states[fault_key].transform.clone()
                                            next_T[:, :3, :3] = predicted_R @ prior_R
                                            states = dict(states)
                                            states[fault_key] = replace(states[fault_key], transform=next_T)
                                        records.append(record)
                print(f'Finished matched 12-window rollouts: {sequence}, start={int(window_indices[0])}, truth {truth_id + 1}/{truths}', flush=True)
    return pd.DataFrame.from_records(records)


In [ ]:
if EVAL_RUN:
    eval_rollouts = evaluate_12_window_rollouts(eval_models)
    eval_rollouts.to_csv(EVAL_DIR / 'rollout_12_window_individual.csv', index=False)
    rollout_summary = eval_rollouts.groupby(['model', 'fault_key', 'axis_type', 'axis_group', 'axis', 'initial_angle_deg', 'sign', 'step'], as_index=False).agg(
        mean_residual_deg=('residual_deg', 'mean'), median_residual_deg=('residual_deg', 'median'), p90_residual_deg=('residual_deg', lambda x: x.quantile(.9)),
        mean_axis_cosine=('axis_cosine', 'mean'), success_10deg=('residual_deg', lambda x: (x < 10).mean()),
        success_30deg=('residual_deg', lambda x: (x < 30).mean()), n=('residual_deg', 'size'))
    rollout_summary.to_csv(EVAL_DIR / 'rollout_12_window_summary_signed.csv', index=False)
    final = eval_rollouts[(eval_rollouts.step == EVAL_ROLLOUT_STEPS) & (eval_rollouts.initial_angle_deg == 179.)]
    print('Final residual after 12 windows from ±179°, grouped by sensor and axis family:')
    display(final.groupby(['model', 'fault_key', 'axis_group']).residual_deg.mean().unstack('axis_group').round(2))
    print('Success rates below 10° at 179°:')
    display(final.groupby(['model', 'fault_key', 'axis_group']).residual_deg.apply(lambda x: (x < 10).mean()).unstack('axis_group').round(3))
    keys = ['sequence', 'start_index', 'truth_id', 'fault_key', 'axis_type', 'axis', 'sign', 'initial_angle_deg', 'step']
    paired_roll = eval_rollouts[eval_rollouts.model.isin(('2.30 source', '61 best'))].pivot(index=keys, columns='model', values='residual_deg').dropna()
    assert len(paired_roll) * 2 == len(eval_rollouts[eval_rollouts.model.isin(('2.30 source', '61 best'))]), 'Unmatched rollout cases.'
    paired_roll['improvement_deg'] = paired_roll['2.30 source'] - paired_roll['61 best']
    paired_roll.reset_index().to_csv(EVAL_DIR / 'rollout_12_window_paired_improvement.csv', index=False)
else:
    print('Rollout comparison disabled.')


In [ ]:
if EVAL_RUN:
    color = {'2.30 source': 'tab:blue', '61 best': 'tab:orange', '2.27 original': 'tab:green'}
    def plot_rollout_grid(metric, initial_angle, title, filename, *, y_lim=None):
        fig, axes = plt.subplots(3, 4, figsize=(18, 11), sharex=True, sharey=True)
        for row, key in enumerate(CALIBRATION_KEYS):
            for col, axis in enumerate(('X', 'Y', 'Z', '3D')):
                ax = axes[row, col]
                subset = eval_rollouts[(eval_rollouts.fault_key == key) & (eval_rollouts.axis_group == axis) & (eval_rollouts.initial_angle_deg == initial_angle)]
                for model_label in eval_models:
                    for sign, style in ((1, '-'), (-1, '--')):
                        values = subset[(subset.model == model_label) & (subset.sign == sign)].groupby('step', as_index=False).agg(mean=(metric, 'mean'))
                        ax.plot(values.step, values['mean'], style, marker='.', color=color[model_label], label=f'{model_label} {"+" if sign == 1 else "−"}')
                if metric == 'residual_deg':
                    ax.axhline(initial_angle, linestyle=':', color='gray', linewidth=1)
                else:
                    ax.axhline(0., linestyle=':', color='gray', linewidth=1)
                ax.set_title(f'{key} / {axis}')
                if y_lim is not None:
                    ax.set_ylim(*y_lim)
                ax.grid(alpha=.25)
                if row == 2:
                    ax.set_xlabel('Applied 5-second update')
                if col == 0:
                    ax.set_ylabel('Mean residual (°)' if metric == 'residual_deg' else 'Signed principal axis cosine')
        axes[0, 0].legend(fontsize=7, ncol=2)
        fig.suptitle(title, fontsize=15)
        fig.tight_layout()
        fig.savefig(EVAL_DIR / filename, dpi=175)
        if SHOW_PLOTS:
            plt.show()
        else:
            plt.close(fig)

    for angle in (30., 90., 150., 179.):
        plot_rollout_grid('residual_deg', angle, f'Matched 12-window feedback from ±{angle:g}°: Cartesian and arbitrary axes', f'rollout_{int(angle)}deg.png', y_lim=(0, 180))
    plot_rollout_grid('axis_cosine', 179., 'Correction-axis alignment throughout 179° rollouts (SO(3) residual is decisive)', 'rollout_179_cosines.png', y_lim=(-1.05, 1.05))

    # Last-step mean and p90. Each 3D group averages several fixed isotropic directions.
    last = eval_rollouts[(eval_rollouts.step == EVAL_ROLLOUT_STEPS) & (eval_rollouts.initial_angle_deg == 179.)]
    tables = {label: last[last.model == label].groupby(['fault_key', 'axis_group']).residual_deg.mean().unstack('axis_group').reindex(index=CALIBRATION_KEYS, columns=('X','Y','Z','3D')) for label in ('2.30 source', '61 best')}
    tables['Improvement'] = tables['2.30 source'] - tables['61 best']
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    for ax, (label, table) in zip(axes, tables.items()):
        raw = table.to_numpy()
        bound = max(10., float(np.nanmax(np.abs(raw))))
        heat = ax.imshow(raw, cmap='RdBu' if label == 'Improvement' else 'viridis_r', vmin=-bound if label == 'Improvement' else 0, vmax=bound if label == 'Improvement' else 180)
        ax.set_xticks(range(4), ('X','Y','Z','3D'))
        ax.set_yticks(range(3), CALIBRATION_KEYS)
        ax.set_title(label)
        for row in range(3):
            for col in range(4):
                ax.text(col, row, f'{table.iloc[row, col]:+.1f}°' if label == 'Improvement' else f'{table.iloc[row, col]:.1f}°', ha='center', va='center', fontsize=9)
        fig.colorbar(heat, ax=ax, shrink=.8)
    fig.suptitle('Mean final 12-window SO(3) residual at ±179°; positive improvement = 61 better')
    fig.tight_layout()
    fig.savefig(EVAL_DIR / 'rollout_179_final_heatmaps.png', dpi=175)
    if SHOW_PLOTS:
        plt.show()
    else:
        plt.close(fig)
    # Inspect failure tails explicitly, including the cases that are almost 180° after rollout.
    rows = []
    for (label, key, family), group in last.groupby(['model', 'fault_key', 'axis_group']):
        rows.append(dict(model=label, fault_key=key, axis_group=family, mean_deg=group.residual_deg.mean(), median_deg=group.residual_deg.median(),
                         p90_deg=group.residual_deg.quantile(.9), frac_below_10deg=(group.residual_deg < 10).mean(),
                         frac_above_150deg=(group.residual_deg > 150).mean(), cases=len(group)))
    rollout_tail = pd.DataFrame(rows)
    rollout_tail.to_csv(EVAL_DIR / 'rollout_179_failure_tails.csv', index=False)
    display(rollout_tail.round(3))
    print('Saved one-step and rollout plots, detailed cases and paired improvements to:', EVAL_DIR)
else:
    print('Rollout plotting disabled.')


## 12. Interpretation and next experiment

- **Success:** the 61 checkpoint improves arbitrary-axis one-step error *and* 12-window rollouts, without a substantial loss on the nine original Cartesian tasks.
- **One-step improves, rollouts do not:** train later on model-generated residual priors using the same single physical window, or add a consistency/contraction regularizer. This is not proof that the 6D representation cannot generalize.
- **Specific directions regress:** inspect near-179° success/failure fractions and the per-case signed cosine distributions. A mean near zero does not by itself imply perpendicular axes near the π cut locus.
- **Weak or inconsistent windows:** compare the two motion sequences, expand rollout starts/truths, and evaluate observability/geometric conditioning before adding Wahba.

**Reproducibility:** keep Notebook 60 002.30 `best.pt` untouched. Notebook 61 saves its new `best.pt` and `last.pt` with AdamW/scheduler/RNG states, plus fixed-probe histories. Never select checkpoints on the untouched Urban38/39 test split.